# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-38/Flyrank_ML_Internship_Projects/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents a single anonymized web page/content item (page_id or equivalent identifier depending on your dataset grain) observed over a specific evaluation window.
Time window: Daily or weekly aggregated search metrics spanning the available observation period in the dataset (typically 30 days or weekly snapshots from the starter dataset/DuckDB warehouse).

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check path"
print("Starter data found successfully!")

# Load data safely
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Total rows: {len(df):,}")
print(f"Columns available: {df.shape[1]}")

Working dir: /flyrank-ml-internship-starter
Starter data found successfully!
Total rows: 30,000
Columns available: 44


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: Numerical and categorical attributes used for prediction (e.g., historical clicks, impressions, average position, content length).

Label: The target outcome we want to forecast or optimize (e.g., future traffic change, ranking shift, or content refresh success).

Context: Identifiers or metadata useful for slicing and grouping, but not direct predictors (e.g., page_id, category, date markers).

Excluded: Raw URLs, specific client/brand names, keywords, and high-cardinality strings (excluded to preserve data privacy and prevent leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verified via programmatic checks: row counts match expected unique keys, missing values are quantified, and window boundaries are continuous without unexplained gaps.

In [5]:
# Check missing values and data types
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary[missing_summary > 0])

# Check duplicate keys if applicable
if 'page_id' in df.columns and 'date' in df.columns:
    duplicates = df.duplicated(subset=['page_id', 'date']).sum()
    print(f"Duplicate (page_id, date) rows: {duplicates}")

Missing values per column:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history: Historical logs may over-represent established pages while lacking sparse data for newly published content.

Platform limits: Search Console metrics are subject to thresholding, sampling limitations, and aggregation noise.

Window overlaps: Sliding windows can introduce autocorrelation, requiring careful validation splits to prevent data leakage.

In [8]:
# Quick distribution check to highlight limits/sparsity
if 'impressions' in df.columns:
    zero_impressions = (df['impressions'] == 0).sum()
    print(f"Rows with zero impressions: {zero_impressions:,} ({zero_impressions / len(df):.2%})")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.